In [ ]:
import pandas as pd
from functools import reduce
import os

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
NHANES_2017_2018_DIR = DATA_DIR / "nhanes" / "2017-2018"

def load(file):
    file_path = NHANES_2017_2018_DIR / file
    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing NHANES file: {file_path}. "
            "Download the required NHANES .xpt files and place them in "
            "data/nhanes/2017-2018/."
        )
    return pd.read_sas(file_path, format="xport")

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
demo  = load("DEMO_J.xpt")
bmx   = load("BMX_J.xpt")

glu   = load("GLU_J.xpt")
ins   = load("INS_J.xpt")

hdl   = load("HDL_J.xpt")
trig  = load("TRIGLY_J.xpt")
tchol = load("TCHOL_J.xpt")

bio   = load("BIOPRO_J.xpt")
crp   = load("HSCRP_J.xpt")
bp    = load("BPX_J.xpt")

In [ ]:
dfs = [demo, bmx, glu, ins, hdl, trig, tchol, bio, crp, bp]

df = reduce(lambda left, right: pd.merge(left, right, on="SEQN", how="outer"), dfs)

In [ ]:
print("Final shape:", df.shape)
print("Missing SEQN:", df["SEQN"].isna().sum())

In [ ]:
df = df.rename(columns={
    "RIDAGEYR": "Age",
    "RIAGENDR": "Sex",
    "RIDRETH1": "Rase",
    
    "LBXGLU": "Glucose",
    "LBXIN": "Insulin",
    
    "LBXTR": "Triglycerides",
    "LBDHDD": "HDL",
    "LBDLDL": "LDL",
    "LBXTC": "Total_Cholesterol",
    
    "LBXSCR": "Creatinine",
    "LBXSUA": "Uric_Acid",
    
    "LBXHSCRP": "CRP",
    "LBXSATSI": "ALT",
    "LBXSASSI": "AST",
    "LBXSGTSI": "GGT",
    
    "LBXSTB": "Bilirubin",
    
    "BMXBMI": "BMI",
    "BMXWAIST": "Waist"
})

In [ ]:
df = df.loc[:, ~df.columns.duplicated()]

In [ ]:
race_dummies = pd.get_dummies(df['Rase'], prefix='Race', drop_first=True)
df = pd.concat([df, race_dummies], axis=1)

In [ ]:
df['Sex_male'] = (df['Sex'] == 1).astype(int)
df['Sex_female'] = (df['Sex'] == 2).astype(int)

In [ ]:
print(f"  Мужчины (Sex_male=1): {df['Sex_male'].sum()}")
print(f"  Женщины (Sex_female=1): {df['Sex_female'].sum()}")

In [ ]:
import numpy as np

df["HOMA_IR"] = (df["Insulin"] * (df["Glucose"]/18)) / 22.5
df["TyG"] = np.log(df["Triglycerides"] * df["Glucose"] / 2)
df["TG_HDL"] = df["Triglycerides"] / df["HDL"]

In [ ]:
print("N final:", len(df))
print("N with CRP:", df["CRP"].notna().sum())
print("N with insulin:", df["Insulin"].notna().sum())

In [ ]:
# Фильтрация по возрасту
df = df[(df["Age"] >= 20) & (df["Age"] <= 80)] 
print(f"Размер после фильтрации по возрасту (>=20): {len(df)}")

distribution check

In [ ]:
cols = [
    "Age","BMI","Waist",
    "Glucose","Insulin","HOMA_IR",
    "Triglycerides","HDL","LDL","Total_Cholesterol",
    "ALT","AST","GGT","Bilirubin",
    "Creatinine","Uric_Acid",
    "CRP"
]

In [ ]:
df["Age"] = df["Age"].mask(df["Age"].abs() < 1e-10)

In [ ]:
df[cols].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).T

In [ ]:
df.to_csv("nhanes_descriptive_stats.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

for col in cols:
    df[col].hist(bins=50)
    plt.title(col)
    plt.show()

In [ ]:
for col in cols:
    q99 = df[col].quantile(0.99)
    max_val = df[col].max()
    print(col, "max/q99 =", round(max_val/q99, 2))

In [ ]:
#винсоризация
skewed_cols = ["Insulin","Triglycerides","CRP","ALT","AST","GGT"]

def winsorize(series, lower=0.01, upper=0.99):
    q_low = series.quantile(lower)
    q_high = series.quantile(upper)
    return series.clip(q_low, q_high)

for col in skewed_cols:
    if col in df.columns:
        df[col] = winsorize(df[col])

In [ ]:
for col in skewed_cols:
    if col in df.columns:
        df["log_" + col] = np.log1p(df[col])

In [ ]:
df[(df["BMI"] < 10) | (df["BMI"] > 80)]

In [ ]:
df = df[(df["BMI"] >= 10) & (df["BMI"] <= 80)]

In [ ]:
df = df[(df["Glucose"] >= 20) & (df["Glucose"] <= 500)]

In [ ]:
df = df[(df["Insulin"] >= 1) & (df["Insulin"] <= 200)]

In [ ]:
df = df[(df["Triglycerides"] >= 10) & (df["Triglycerides"] <= 2000)]

In [ ]:
df = df[(df["CRP"] >= 0.1) & (df["CRP"] <= 100)]

In [ ]:
df[cols].isna().mean().sort_values(ascending=False)

In [ ]:
print(f"После фильтрации: N = {len(df)}")
print(df["Insulin"].describe())

In [ ]:
df[cols].corr()

In [ ]:
import seaborn as sns
# Таблица корреляций
cols = [
    "BMI","Waist",
    "Glucose","Insulin","HOMA_IR",
    "Triglycerides","HDL","LDL","Total_Cholesterol",
    "ALT","AST","GGT",
    "Creatinine","Uric_Acid","CRP", 'Bilirubin',
]

corr_matrix = df[cols].corr(method='spearman')
# Тепловая карта
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()
# Таблица корреляций
print(corr_matrix)

In [ ]:
import pingouin as pi
# Partial Spearman: контроль на Sex, Age, BMI
cols = [
    "Waist",
    "Glucose","Insulin","HOMA_IR",
    "Triglycerides","HDL","LDL","Total_Cholesterol",
    "ALT","AST","GGT",
    "Creatinine","Uric_Acid","CRP","Bilirubin",
]
confounders = ["Sex","Age"] + [col for col in race_dummies.columns]
# 1. Обычный Spearman (baseline)
spearman_base = df[cols].corr(method='spearman')
print("=== Regular Spearman ===")
plt.figure(figsize=(12,10))
import seaborn as sns
sns.heatmap(spearman_base, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5, cbar_kws={"shrink":0.8})
plt.title("Spearman Correlation (unadjusted)")
plt.tight_layout()
plt.show()
# 2. Partial Spearman (контроль Sex, Age, BMI)
import warnings
warnings.filterwarnings('ignore')
partial_mat = pd.DataFrame(index=cols, columns=cols, dtype=float)
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        if i >= j:
            continue
        covar = [c for c in confounders if c not in [c1, c2]]
        data = df[[c1, c2] + covar].dropna()
        if len(data) < 10:
            partial_mat.loc[c1, c2] = partial_mat.loc[c2, c1] = float('nan')
            continue
        res = pi.partial_corr(data=data, x=c1, y=c2, covar=covar, method='spearman')
        rho = res['r'].iloc[0]
        partial_mat.loc[c1, c2] = partial_mat.loc[c2, c1] = rho
plt.figure(figsize=(12,10))
sns.heatmap(partial_mat.astype(float), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5, cbar_kws={"shrink":0.8})
plt.title("Partial Spearman (adjusted for Sex, Age, Rase)")
plt.tight_layout()
plt.show()
print()
print("=== Partial Spearman Matrix ===")
print(partial_mat.astype(float).round(3))
# 3. Таблица пар (rho + p-value), только significant
print()
print("=== Significant Partial Spearman (p < 0.05) ===")
results = []
for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        if i >= j:
            continue
        covar = [c for c in confounders if c not in [c1, c2]]
        data = df[[c1, c2] + covar].dropna()
        if len(data) < 10:
            continue
        res = pi.partial_corr(data=data, x=c1, y=c2, covar=covar, method='spearman')
        rho = res['r'].iloc[0]
        p = res['p_val'].iloc[0]
        results.append({'Var1': c1, 'Var2': c2, 'rho': rho, 'p': p, 'N': len(data), 'sig': p < 0.05})
res_df = pd.DataFrame(results).sort_values('p')
res_df['rho'] = res_df['rho'].round(3)
res_df['p'] = res_df['p'].round(4)
print(res_df[res_df['sig']].to_string(index=False))
print()
print(f"Total pairs tested: {len(res_df)}, significant: {(res_df['p']<0.05).sum()}")


In [ ]:
df.to_csv("nhanes_metabolic_features.csv", index=False)

In [ ]:
from scipy.stats import shapiro

for col in ["Glucose","Insulin","Triglycerides","CRP"]:
    stat, p = shapiro(df[col].dropna())
    print(col, p)

Метаболический синдром

In [ ]:
# Drug therapy flags (MetS criterion = lab value OR drug intake)
# Build drug name -> RXDDRGID mapping from RXQ_DRUG

drug_dict = load('RXQ_DRUG.xpt')
rxq = load('RXQ_RX_J.xpt')

# Decode byte strings and normalize
drug_dict['RXDDRUG_clean'] = drug_dict['RXDDRUG'].apply(
    lambda x: x.decode('latin-1').strip().upper() if isinstance(x, bytes) else (str(x).strip().upper() if pd.notna(x) else '')
)
drug_dict['RXDDRGID_clean'] = drug_dict['RXDDRGID'].apply(
    lambda x: x.decode('latin-1').strip() if isinstance(x, bytes) else (str(x).strip() if pd.notna(x) else '')
)

rxq['RXDDRUG_clean'] = rxq['RXDDRUG'].apply(
    lambda x: x.decode('latin-1').strip().upper() if isinstance(x, bytes) else (str(x).strip().upper() if pd.notna(x) else '')
)
rxq['RXDDRGID_clean'] = rxq['RXDDRGID'].apply(
    lambda x: x.decode('latin-1').strip() if isinstance(x, bytes) else (str(x).strip() if pd.notna(x) else '')
)

# Map drug names to their RXDDRGID sets (case-insensitive search)
target_drugs = {
    'metformin': ['METFORMIN'],
    'insulin': ['INSULIN'],
    'glipizide': ['GLIPIZIDE'],
    'glyburide': ['GLYBURIDE'],
    'lisinopril': ['LISINOPRIL'],
    'amlodipine': ['AMLODIPINE'],
    'losartan': ['LOSARTAN'],
    'atenolol': ['ATENOLOL'],
    'atorvastatin': ['ATORVASTATIN'],
    'simvastatin': ['SIMVASTATIN'],
    'rosuvastatin': ['ROSUVASTATIN'],
    'fenofibrate': ['FENOFIBRATE'],
}

# For each drug name, collect all RXDDRGID codes that contain it
drug_to_ids = {}
for drug, names in target_drugs.items():
    ids = set()
    for _, row in drug_dict.iterrows():
        drug_name = row['RXDDRUG_clean']
        for n in names:
            if n in drug_name:
                ids.add(row['RXDDRGID_clean'])
    drug_to_ids[drug] = ids

# Map to categories
cat_drugs = {
    'antidiabetic': ['metformin', 'insulin', 'glipizide', 'glyburide'],
    'antihypertensive': ['lisinopril', 'amlodipine', 'losartan', 'atenolol'],
    'lipid_lowering': ['atorvastatin', 'simvastatin', 'rosuvastatin', 'fenofibrate'],
}

cat_ids = {}
for cat, drugs in cat_drugs.items():
    ids = set()
    for d in drugs:
        ids.update(drug_to_ids[d])
    cat_ids[cat] = ids

print('Drug IDs per category:')
for cat, ids in cat_ids.items():
    print(f'  {cat}: {len(ids)} codes')

# Filter RXQ rows that match any target drug
rxq_filtered = rxq[rxq['RXDDRGID_clean'].isin(set().union(*cat_ids.values()))].copy()
print(f'RXQ rows with target drugs: {len(rxq_filtered)}')
print(f'Unique participants: {rxq_filtered["SEQN"].nunique()}')

# Build per-participant flags
therapy = rxq_filtered.groupby('SEQN').agg(
    on_antidiabetic=('RXDDRGID_clean', lambda x: any(x.isin(cat_ids['antidiabetic']))),
    on_antihypertensive=('RXDDRGID_clean', lambda x: any(x.isin(cat_ids['antihypertensive']))),
    on_lipid_lowering=('RXDDRGID_clean', lambda x: any(x.isin(cat_ids['lipid_lowering']))),
).reset_index()

therapy['on_antidiabetic'] = therapy['on_antidiabetic'].astype(int)
therapy['on_antihypertensive'] = therapy['on_antihypertensive'].astype(int)
therapy['on_lipid_lowering'] = therapy['on_lipid_lowering'].astype(int)

print(f'Therapy participants: {len(therapy)}')
print(therapy[['on_antidiabetic','on_antihypertensive','on_lipid_lowering']].sum())

# Merge into df
df = df.merge(therapy, on='SEQN', how='left')
df['on_antidiabetic'] = df['on_antidiabetic'].fillna(0).astype(int)
df['on_antihypertensive'] = df['on_antihypertensive'].fillna(0).astype(int)
df['on_lipid_lowering'] = df['on_lipid_lowering'].fillna(0).astype(int)

print(f'Total participants: {len(df)}')
print(df[['on_antidiabetic','on_antihypertensive','on_lipid_lowering']].sum())


In [ ]:
# МЕТАБОЛИЧЕСКИЙ СИНДРОМ (с учётом лекарственной терапии)

# Пол: 1 = male, 2 = female
male = df["Sex"] == 1
female = df["Sex"] == 2

# 1. Ожирение (BMI >= 30)
# Лекарств от ожирения нет, поэтому только по BMI
df["crit_bmi"] = (df["BMI"] >= 30).astype(int)

# 2. Гипертриглицеридемия (TG >= 150 мг/дл или on_lipid_lowering)
# Если пациент принимает статины/фибраты -> критерий засчитывается
df["crit_tg"] = (
    (df["Triglycerides"] >= 150) | 
    (df["on_lipid_lowering"] == 1)
).astype(int)

# 3. Низкий ХС-ЛПВП (HDL)
# Если пациент принимает статины/фибраты -> критерий засчитывается
df["crit_hdl"] = (
    (
        ((male) & (df["HDL"] < 40)) |
        ((female) & (df["HDL"] < 50))
    ) | 
    (df["on_lipid_lowering"] == 1)
).astype(int)

# 4. Гипергликемия (Glucose >= 100 мг/дл или on_antidiabetic)
df["crit_glu"] = (
    (df["Glucose"] >= 100) | 
    (df["on_antidiabetic"] == 1)
).astype(int)

# 5. Артериальная гипертензия (SBP >= 130 или DBP >= 85 или on_antihypertensive)
# Сначала вычисляем среднее давление
df["SBP"] = df[["BPXSY1", "BPXSY2", "BPXSY3"]].mean(axis=1)
df["DBP"] = df[["BPXDI1", "BPXDI2", "BPXDI3"]].mean(axis=1)

df["crit_bp"] = (
    (df["SBP"] >= 130) | 
    (df["DBP"] >= 85) | 
    (df["on_antihypertensive"] == 1)
).astype(int)

# Суммируем критерии
df["MetS_score"] = df[
    ["crit_bmi", "crit_tg", "crit_hdl", "crit_glu", "crit_bp"]
].sum(axis=1)

# Финальная метка MetS (3 и более критерия)
df["MetS"] = (df["MetS_score"] >= 3).astype(int)

# Проверка
print("Распределение MetS (с учётом терапии):")
print(df["MetS"].value_counts())
print(f"Доля MetS: {df['MetS'].mean():.2%}")

# Проверка: у скольких пациентов MetS поставили только из-за лекарств?

print("\nПациенты с MetS ТОЛЬКО из-за лекарств (лаборатория в норме):")
print(f"  Антидиабетики: {((df['crit_glu']==1) & (df['Glucose']<100) & (df['on_antidiabetic']==1)).sum()}")
print(f"  Антигипертензивные: {((df['crit_bp']==1) & (df['SBP']<130) & (df['DBP']<85) & (df['on_antihypertensive']==1)).sum()}")
print(f"  Гиполипидемические: {((df['crit_tg']==1) & (df['Triglycerides']<150) & (df['on_lipid_lowering']==1)).sum()}")

In [ ]:
# Приведение единиц измерения к международным

# Создаем колонки с новыми единицами
df['Glucose_mmolL'] = df['Glucose'] / 18.016
df['Insulin_mIU_L'] = df['Insulin']  # Единицы уже mIU/L, но переименуем для ясности
df['HOMA_IR_mmol_mIU'] = df['HOMA_IR']  # Формула использует mmol/L, но мы пересчитаем для чистоты
df['Triglycerides_mmolL'] = df['Triglycerides'] / 88.57
df['HDL_mmolL'] = df['HDL'] / 38.67
df['LDL_mmolL'] = df['LDL'] / 38.67
df['Total_Cholesterol_mmolL'] = df['Total_Cholesterol'] / 38.67

# Для биомаркеров
df['Creatinine_umolL'] = df['Creatinine'] * 88.4
df['Uric_Acid_umolL'] = df['Uric_Acid'] * 59.48

# Пересчитаем HOMA-IR уже с пересчитанной глюкозой (для абсолютной точности)
df['HOMA_IR_mmol_mIU'] = (df['Insulin_mIU_L'] * df['Glucose_mmolL']) / 22.5

print("Единицы измерения приведены к международному формату (ммоль/л, мкмоль/л).")
print("Новые колонки: Glucose_mmolL, Insulin_mIU_L, Triglycerides_mmolL, etc.")

In [ ]:
# Balance groups by sex (1=male, 2=female)
# For each MetS class, equalize M/F counts using min group
print('Before balancing:')
print(df.groupby(['MetS','Sex']).size().unstack(fill_value=0))

# Create balanced df
balanced_rows = []
for mets_val in [0, 1]:
    for sex_val in [1, 2]:
        subset = df[(df['MetS'] == mets_val) & (df['Sex'] == sex_val)]
        balanced_rows.append(subset)

min_counts = {}
for mets_val in [0, 1]:
    n_m = len(balanced_rows[[i for i in range(4) if i%2==0][mets_val]])  # male index
    n_f = len(balanced_rows[[i for i in range(4) if i%2==1][mets_val]])  # female index
    min_counts[mets_val] = min(n_m, n_f)
    print(f'MetS={mets_val}: M={n_m}, F={n_f}, min={min_counts[mets_val]}')

# Sample to min per MetS class
new_rows = []
for mets_val in [0, 1]:
    for sex_val in [1, 2]:
        subset = df[(df['MetS'] == mets_val) & (df['Sex'] == sex_val)]
        new_rows.append(subset.sample(n=min_counts[mets_val], random_state=42))

df = pd.concat(new_rows, ignore_index=True)

print('After balancing:')
print(df.groupby(['MetS','Sex']).size().unstack(fill_value=0))
print(f'Total N: {len(df)}')
print(df['MetS'].value_counts())


In [ ]:
df["MetS"].mean()

In [ ]:
from scipy.stats import mannwhitneyu

for col in ["HOMA_IR","ALT","GGT","CRP","Uric_Acid"]:
    a = df[df["MetS"] == 1][col].dropna()
    b = df[df["MetS"] == 0][col].dropna()
    
    stat, p = mannwhitneyu(a, b)
    print(col, p)

In [ ]:
model_cols = [
    "MetS",
    "Age", "Sex",
    "BMI", "Waist",
    "Glucose", "Insulin",
    "Triglycerides", "HDL",
    "ALT", "AST", "GGT",
    "CRP"
]

df_model = df[model_cols].dropna()

print("N for model:", df_model.shape[0])

ML

In [ ]:
diag = [
    "BMI",
    "Triglycerides",
    "HDL",
    "Glucose",
    "SBP",
    "DBP"
]

In [ ]:
non_diag = [
    "Age", "Sex",
    "ALT", "AST", "GGT",
    "CRP",
    "Uric_Acid",
    "Creatinine",
    "Insulin"
]

In [ ]:
features_A = ["Age","Sex"] + diag
features_B = ["Age","Sex"] + diag + non_diag
features_C = ["Age","Sex"] + non_diag

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

def run_models(df, features):
    df_model = df[features + ["MetS"]].dropna()

    print("N =", df_model.shape[0])

    X = df_model[features]
    y = df_model["MetS"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    results = {}

    #Logistic Regression
    from sklearn.linear_model import LogisticRegression
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    logreg = LogisticRegression(max_iter=5000)
    logreg.fit(X_train_scaled, y_train)

    y_pred = logreg.predict_proba(X_test_scaled)[:,1]
    results["Logit"] = roc_auc_score(y_test, y_pred)

    #Random Forest
    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=300, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict_proba(X_test)[:,1]
    results["RF"] = roc_auc_score(y_test, y_pred)

    #XGBoost
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
    xgb.fit(X_train, y_train)
    y_pred = xgb.predict_proba(X_test)[:,1]
    results["XGB"] = roc_auc_score(y_test, y_pred)

    return results

In [ ]:
features_B = list(dict.fromkeys(features_B))
features_C = list(dict.fromkeys(features_C))

In [ ]:
print("Model A (diagnostic):")
res_A = run_models(df, features_A)
print(res_A)

print("\nModel B (all):")
res_B = run_models(df, features_B)
print(res_B)

print("\nModel C (non-diagnostic):")
res_C = run_models(df, features_C)
print(res_C)

In [ ]:
from sklearn.metrics import roc_auc_score

def bootstrap_metric(y_true, y_prob, metric_func, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    scores = []
    
    y_true = y_true.reset_index(drop=True)
    
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        
        y_t = y_true.iloc[idx]
        y_p = y_prob[idx]
        
        # для AUC важно
        if metric_func.__name__ == "roc_auc_score":
            if len(np.unique(y_t)) < 2:
                continue
        
        score = metric_func(y_t, y_p)
        scores.append(score)
    
    low = np.percentile(scores, 2.5)
    high = np.percentile(scores, 97.5)
    
    return low, high

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns


def find_optimal_threshold(y_true, y_prob, metric="f1"):
    """Find optimal threshold for given metric."""
    best_thresh, best_score = 0.5, 0
    for thresh in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob >= thresh).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "youden":
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            score = tpr - fpr
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh, best_score


def evaluate_models(df, features, target="MetS", title="Model"):
    X = df[features].copy()
    y = df[target].copy()
    
    X = X.select_dtypes(include=[np.number])
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    y_test = y_test.reset_index(drop=True)
    
    models = {
        "LogReg": Pipeline([("scaler", StandardScaler()), ("logit", LogisticRegression(max_iter=5000, class_weight="balanced"))]),
        "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
        "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42,
                             scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]))
    }
    
    results = []
    
    #ROC-КРИВЫЕ НА ОДНОМ ГРАФИКЕ
    plt.figure(figsize=(7,6))
    
    for name, model in models.items():
        if name == "RF":
            X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
            model.fit(X_train_sm, y_train_sm)
        elif name == "XGB":
            X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train, y_train)
            model.fit(X_train_sm, y_train_sm)
        else:
            model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:,1]
        y_pred = (y_prob > 0.5).astype(int)
    
        #ROC
        auc = roc_auc_score(y_test, y_prob)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    
        #bootstrap CI
        auc_low, auc_high = bootstrap_metric(y_test, y_prob, roc_auc_score)

        acc_low, acc_high = bootstrap_metric(y_test, y_pred, accuracy_score)
        prec_low, prec_high = bootstrap_metric(y_test, y_pred, precision_score)
        rec_low, rec_high = bootstrap_metric(y_test, y_pred, recall_score)
        f1_low, f1_high = bootstrap_metric(y_test, y_pred, f1_score)
    
        #оптимальный порог (F1)
        opt_thresh, opt_score = find_optimal_threshold(y_test.values, y_prob, metric="f1")
        y_pred_opt = (y_prob >= opt_thresh).astype(int)

        acc_opt = accuracy_score(y_test, y_pred_opt)
        prec_opt = precision_score(y_test, y_pred_opt, zero_division=0)
        rec_opt = recall_score(y_test, y_pred_opt, zero_division=0)
        f1_opt = f1_score(y_test, y_pred_opt, zero_division=0)

        #сохраняем (с оптимальным порогом)
        results.append({
            "Scenario": title,
            "Model": name,
            "AUC": f"{auc:.3f} ({auc_low:.3f}-{auc_high:.3f})",
            "Accuracy": f"{acc_opt:.3f}",
            "Precision": f"{prec_opt:.3f}",
            "Recall": f"{rec_opt:.3f}",
            "F1": f"{f1_opt:.3f}"
        })
    
    plt.plot([0, 1], [0, 1], 'k--', label="Random (AUC=0.5)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves - {title}")
    plt.legend()
    plt.show()
    
    # МАТРИЦЫ ОШИБОК
    for name, model in models.items():
        model.fit(X_train, y_train) 
        y_pred = (model.predict_proba(X_test)[:,1] > 0.5).astype(int)
        
        plt.figure()
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"{title} - {name} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.show()
    y_probs = {name: model.predict_proba(X_test)[:,1] for name, model in models.items()}
    return pd.DataFrame(results), models, X_test, y_test.reset_index(drop=True), y_probs

In [ ]:
features_A = ["BMI","Glucose","Triglycerides","HDL","SBP","DBP"]
res_A, models_A, X_test_A, y_test_A, y_probs_A = evaluate_models(df, features_A, title="Diagnostic")
print(res_A)

In [ ]:
features_B = [
    "Age","Sex","BMI","Glucose","Insulin","Triglycerides",
    "HDL","ALT","AST","GGT","CRP"
]

res_B, models_B, X_test_B, y_test_B, y_probs_B = evaluate_models(df, features_B, title="All")
print(res_B)

In [ ]:
features_C = [
    "Age","Sex","ALT","AST","GGT","CRP","Uric_Acid","Creatinine"
]

res_C,  models_C, X_test_C, y_test_C, y_probs_C = evaluate_models(df, features_C, title="Non-diagnostic")
print(res_C)

DeLong test

In [ ]:
from pauc import ROC, compare
scenarios = [
    ("A (Diagnostic)", y_test_A, y_probs_A),
    ("B (All)", y_test_B, y_probs_B),
    ("C (Non-diagnostic)", y_test_C, y_probs_C),
]
for scenario_name, y_test, y_probs in scenarios:
    print(f"\n{'='*60}")
    print(f"Scenario {scenario_name}")
    print(f"{'='*60}")
    
    models = list(y_probs.keys())
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            m1, m2 = models[i], models[j]
            roc1 = ROC(y_test, y_probs[m1])
            roc2 = ROC(y_test, y_probs[m2])
            result = compare(roc1, roc2, method="delong")
            print(f"{m1} vs {m2}: p={result.p_value:.4f}, AUC={roc1.auc:.4f} vs {roc2.auc:.4f}")

SHAP XGBoost

In [ ]:
import shap

model = models_A["XGB"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_A)

shap.summary_plot(shap_values, X_test_A)

In [ ]:
import shap
model = models_A["XGB"]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_A)
# Таблица средних SHAP значений
shap_df = pd.DataFrame({
    'feature': X_test_A.columns,
    'shap_mean': np.abs(shap_values).mean(axis=0),
    'shap_std': np.std(shap_values, axis=0)
}).sort_values('shap_mean', ascending=False)
print("Scenario A (XGB):")
print(shap_df)

In [ ]:
model = models_B["XGB"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_B)

shap.summary_plot(shap_values, X_test_B)

In [ ]:
import shap
model = models_B["XGB"]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_B)
# Таблица средних SHAP значений
shap_df = pd.DataFrame({
    'feature': X_test_B.columns,
    'shap_mean': np.abs(shap_values).mean(axis=0),
    'shap_std': np.std(shap_values, axis=0)
}).sort_values('shap_mean', ascending=False)
print("Scenario B (XGB):")
print(shap_df)

In [ ]:
model = models_C["XGB"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_C)

shap.summary_plot(shap_values, X_test_C)

In [ ]:
import shap
model = models_C["XGB"]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_C)
# Таблица средних SHAP значений
shap_df = pd.DataFrame({
    'feature': X_test_C.columns,
    'shap_mean': np.abs(shap_values).mean(axis=0),
    'shap_std': np.std(shap_values, axis=0)
}).sort_values('shap_mean', ascending=False)
print("Scenario C (XGB):")
print(shap_df)

SHAP Random Forest

In [ ]:
import shap

In [ ]:
print(X_test_A.shape)
print(X_test_A.columns)

In [ ]:
X = df[features_A].copy()
y = df["MetS"].copy()

X = X.select_dtypes(include=[np.number])
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

from sklearn.model_selection import train_test_split

X_train, X_test_A, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
model_rf = RandomForestClassifier(n_estimators=300, random_state=42)
model_rf.fit(X_train, y_train)

In [ ]:
explainer = shap.Explainer(model_rf, X_train)

shap_values = explainer(
    X_test_A,
    check_additivity=False   # ← ВОТ ЭТО КЛЮЧЕВОЕ
)
shap_vals_class1 = shap_values[:, :, 1]

shap.summary_plot(shap_vals_class1, X_test_A)

In [ ]:
explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X_test_A)

# если список → RF классика
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values[:, :, 1]

# таблица
shap_df = pd.DataFrame({
    "Feature": X_test_A.columns,
    "SHAP_importance": np.abs(shap_vals).mean(axis=0)
}).sort_values(by="SHAP_importance", ascending=False)

print(shap_df)

In [ ]:
explainer = shap.Explainer(model_rf, X_train)

shap_values = explainer(
    X_test_B,
    check_additivity=False
)
shap_vals_class1 = shap_values[:, :, 1]

shap.summary_plot(shap_vals_class1, X_test_B)

In [ ]:
explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X_test_B)

# если список → RF классика
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values[:, :, 1]

# таблица
shap_df = pd.DataFrame({
    "Feature": X_test_B.columns,
    "SHAP_importance": np.abs(shap_vals).mean(axis=0)
}).sort_values(by="SHAP_importance", ascending=False)

print(shap_df)

In [ ]:
explainer = shap.Explainer(model_rf, X_train)

shap_values = explainer(
    X_test_C,
    check_additivity=False
)
shap_vals_class1 = shap_values[:, :, 1]

shap.summary_plot(shap_vals_class1, X_test_C)

In [ ]:
explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X_test_C)

# если список → RF классика
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values[:, :, 1]

# таблица
shap_df = pd.DataFrame({
    "Feature": X_test_C.columns,
    "SHAP_importance": np.abs(shap_vals).mean(axis=0)
}).sort_values(by="SHAP_importance", ascending=False)

print(shap_df)

Статистика

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, chi2_contingency

def make_table1(df, target="MetS"):
    
    continuous = [
        "Age","BMI","Waist","Glucose","Triglycerides",
        "HDL","ALT","AST","GGT","CRP","Uric_Acid"
    ]
    
    categorical = ["Sex"]
    
    table = []
    p_values = []  # для FDR
    
    # Continuous variables
    for var in continuous:
        
        g0 = df[df[target]==0][var].dropna()
        g1 = df[df[target]==1][var].dropna()
        
        # Mann–Whitney
        stat, p = mannwhitneyu(g0, g1, alternative="two-sided")
        p_values.append(p)
        
        def fmt(x):
            return f"{x.median():.2f} [{x.quantile(0.25):.2f}–{x.quantile(0.75):.2f}]"
        
        table.append({
            "Variable": var,
            "Non-MetS": fmt(g0),
            "MetS": fmt(g1),
            "p-value": p
        })
    
    # Categorical variables
    for var in categorical:
        
        ct = pd.crosstab(df[var], df[target])
        chi2, p, _, _ = chi2_contingency(ct)
        p_values.append(p)
        
        g0 = df[df[target]==0][var]
        g1 = df[df[target]==1][var]
        
        def fmt_cat(x, value):
            return f"{(x==value).sum()} ({(x==value).mean()*100:.1f}%)"
        
        table.append({
            "Variable": f"{var} (Male)",
            "Non-MetS": fmt_cat(g0, 1),
            "MetS": fmt_cat(g1, 1),
            "p-value": p
        })
        
        table.append({
            "Variable": f"{var} (Female)",
            "Non-MetS": fmt_cat(g0, 2),
            "MetS": fmt_cat(g1, 2),
            "p-value": None  # chi2 только для Male
        })
    
    # FDR correction (Benjamini-Hochberg)
    _, p_adj, _, _ = multipletests(p_values, method='fdr_bh')
    
    # Добавляем скорректированные p-values в таблицу
    p_idx = 0
    for row in table:
        if row["p-value"] is not None:
            row["p_adj"] = p_adj[p_idx]
            p_idx += 1
        else:
            row["p_adj"] = None
    
    # Форматирование для вывода
    def fmt_p(p):
        if p is None:
            return ""
        if p < 0.001:
            return "<0.001"
        elif p < 0.01:
            return f"{p:.3f}"
        elif p < 0.05:
            return f"{p:.3f}"
        else:
            return f"{p:.3f}"
    
    # Создаем итоговый DataFrame для вывода
    result_df = pd.DataFrame(table)
    
    # Форматируем p-values
    result_df["p-value"] = result_df["p-value"].apply(lambda x: fmt_p(x) if x is not None else "")
    result_df["p_adj"] = result_df["p_adj"].apply(lambda x: fmt_p(x) if x is not None else "")
    
    # Переименовываем колонки для финальной таблицы
    result_df = result_df.rename(columns={
        "Variable": "Characteristic",
        "Non-MetS": f"Non-{target} (N={len(df[df[target]==0])})",
        "MetS": f"{target} (N={len(df[df[target]==1])})",
        "p-value": "P-value",
        "p_adj": "P-value (FDR adjusted)"
    })
    
    return result_df

In [ ]:
table1 = make_table1(df)
print(table1)

10-кратная кросс-валидация

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
## Cross-Validation (k=10)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score
import numpy as np
import matplotlib.pyplot as plt

def find_optimal_threshold(y_true, y_prob, metric="f1"):
    best_thresh, best_score = 0.5, 0
    for thresh in np.arange(0.1, 0.9, 0.02):
        y_pred = (y_prob >= thresh).astype(int)
        if metric == "f1":
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == "youden":
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            score = tpr - fpr
        if score > best_score:
            best_score = score
            best_thresh = thresh
    return best_thresh, best_score


def evaluate_models_cv(df, features, target="MetS", title="Model", n_splits=10):
    X = df[features].copy()
    y = df[target].copy()
    
    X = X.select_dtypes(include=[np.number])
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    
    models = {
        "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced"),
        "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
        "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42,
                           scale_pos_weight=len(y[y==0])/max(1,len(y[y==1])))
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []
    
    plt.figure(figsize=(7,6))
    
    for name, model in models.items():
        if name == "LogReg":
            X_used = X_scaled
        else:
            X_used = X
        
        y_prob = cross_val_predict(model, X_used, y, cv=skf, method="predict_proba")[:, 1]
        y_pred = (y_prob > 0.5).astype(int)
        
        auc = roc_auc_score(y, y_prob)
        fpr, tpr, _ = roc_curve(y, y_prob)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
        
        # CV для std
        aucs = []
        for train_idx, val_idx in skf.split(X_used, y):
            X_tr, X_val = X_used.iloc[train_idx], X_used.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            
            model_cv = {
                "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced"),
                "RF": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
                "XGB": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42)
            }[name]
            
            if name == "LogReg":
                scaler_cv = StandardScaler()
                X_tr = scaler_cv.fit_transform(X_tr)
                X_val = scaler_cv.transform(X_val)
                model_cv.fit(X_tr, y_tr)
            elif name == "RF":
                model_cv.fit(X_tr, y_tr)
            elif name == "XGB":
                model_cv.fit(X_tr, y_tr)
            else:
                model_cv.fit(X_tr, y_tr)
                
            y_prob_cv = model_cv.predict_proba(X_val)[:, 1]
            aucs.append(roc_auc_score(y_val, y_prob_cv))
        
        auc_mean, auc_std = np.mean(aucs), np.std(aucs)
        
        opt_thresh, opt_f1 = find_optimal_threshold(y.values, y_prob, metric="f1")
        y_pred = (y_prob >= opt_thresh).astype(int)
        prec = precision_score(y, y_pred, zero_division=0)
        rec = recall_score(y, y_pred, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        results.append({
            "Scenario": title,
            "Model": name,
            "AUC": f"{auc_mean:.3f}±{auc_std:.3f}",
            "Precision": f"{prec:.3f}",
            "Recall": f"{rec:.3f}",
            "F1": f"{f1:.3f}"
        })
    
    plt.plot([0, 1], [0, 1], 'k--', label="Random (AUC=0.5)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves (CV) - {title}")
    plt.legend()
    plt.show()
    
    return pd.DataFrame(results)

print("="*60)
print("CROSS-VALIDATION (k=10)")
print("="*60)
features_A = ["BMI","Glucose","Triglycerides","HDL","SBP","DBP"]
features_B = ["Age","Sex","BMI","Glucose","Insulin","Triglycerides","HDL","ALT","AST","GGT","CRP"]
features_C = ["Age","Sex","ALT","AST","GGT","CRP","Uric_Acid","Creatinine"]
cv_A = evaluate_models_cv(df, features_A, title="A (Diagnostic)")
print("\nScenario A (Diagnostic):")
print(cv_A)
cv_B = evaluate_models_cv(df, features_B, title="B (All)")
print("\nScenario B (All):")
print(cv_B)
cv_C = evaluate_models_cv(df, features_C, title="C (Non-diagnostic)")
print("\nScenario C (Non-diagnostic):")
print(cv_C)
